# 01 — Exploratory data analysis

**Project:** Employee Performance Predictor using Data Analytics (educational simulation)

**Objective:** Predict the **next-cycle** performance band (High / Medium / Low) from simulated review-cycle signals.

This notebook is **Phase 3 only**: data quality and EDA. It does not train a model and does not make promotion, termination, or disciplinary recommendations.

**Important:** The table is 100% synthetic. Charts and correlations describe this file. They are **not** evidence about real employees.


## Dataset overview

| Item | Detail |
| --- | --- |
| Raw file | data/raw/employee_performance_synthetic.csv |
| Clean file | data/processed/employee_performance_clean.csv (written by python -m src.eda) |
| Grain | One simulated employee **review cycle** |
| Expected shape | 2,000 rows, 26 columns, 1,000 employees, 2 cycles each |

employee_id is an identifier. Do **not** treat it as a performance signal or include it in correlations as if it were a feature.


In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..') if Path('data').exists() is False else Path('.')
# Prefer running with the project root as the working directory.
for candidate in (Path('.'), Path('..')):
    raw_try = candidate / 'data' / 'raw' / 'employee_performance_synthetic.csv'
    if raw_try.exists():
        PROJECT_ROOT = candidate
        break

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'employee_performance_synthetic.csv'
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'employee_performance_clean.csv'

BAND_ORDER = ['Low', 'Medium', 'High']
print('Project root:', PROJECT_ROOT.resolve())
print('Raw exists:', RAW_PATH.exists())
print('Clean exists:', CLEAN_PATH.exists())


In [ ]:
raw = pd.read_csv(RAW_PATH, dtype={'employee_id': 'string'})
clean = pd.read_csv(CLEAN_PATH, dtype={'employee_id': 'string'}) if CLEAN_PATH.exists() else raw
df = clean.copy()
df['perf_band_next'] = pd.Categorical(df['perf_band_next'], categories=BAND_ORDER, ordered=True)
df.head()


## Shape, types, missing values, duplicates, descriptive statistics


In [ ]:
print('Clean shape:', df.shape)
print('Raw shape:', raw.shape)
print()
print(df.dtypes)
print()
print('Missing cells:', int(df.isna().sum().sum()))
print('Duplicate rows:', int(df.duplicated().sum()))
print('Duplicate employee+cycle:', int(df.duplicated(subset=['employee_id', 'review_cycle']).sum()))
print('Unique employees:', df['employee_id'].nunique())
print(df.groupby('employee_id').size().value_counts().sort_index())


In [ ]:
df.describe().T.round(3)


## Target-class distribution

Computed on the clean file when this notebook is executed. Reference values from the Phase 3 run: Medium 743 (37.15%), High 671 (33.55%), Low 586 (29.30%).


In [ ]:
counts = df['perf_band_next'].value_counts().reindex(BAND_ORDER)
perc = (counts / counts.sum() * 100).round(2)
target_table = pd.DataFrame({'count': counts, 'percent': perc})
print(target_table)
sns.countplot(data=df, x='perf_band_next', order=BAND_ORDER)
plt.title('Next-cycle performance band (synthetic)')
plt.xlabel('perf_band_next')
plt.ylabel('count')
plt.show()


## Numerical feature distributions

Login hours and sick days are plotted for contrast: they were **not** used to generate the label.


In [ ]:
dist_cols = [
    'on_time_delivery_rate', 'avg_task_delay_days', 'bug_count',
    'code_review_score', 'story_points_completed', 'training_hours',
    'peer_feedback_score', 'kudos_count', 'sick_days', 'avg_login_hours',
]
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.flatten()
for ax, col in zip(axes, dist_cols):
    sns.histplot(df[col], bins=20, kde=True, ax=ax)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
fig.suptitle('Numeric distributions (synthetic)')
fig.tight_layout()
plt.show()


## Target-band comparisons for selected numeric features

Overlap is expected. A clean separation would be a warning that labels are too easy.


In [ ]:
signal_cols = [
    'on_time_delivery_rate', 'avg_task_delay_days', 'bug_count',
    'training_hours', 'peer_feedback_score', 'previous_manager_score',
]
print(df.groupby('perf_band_next', observed=True)[signal_cols + ['sick_days', 'avg_login_hours']].mean().round(3))
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, col in zip(axes, signal_cols):
    sns.boxplot(data=df, x='perf_band_next', y=col, order=BAND_ORDER, ax=ax)
    ax.set_title(col)
fig.suptitle('Signals by next-cycle band (synthetic)')
fig.tight_layout()
plt.show()


## Categorical summaries: department, job level, education

These fields were **not** used to generate perf_band_next. Percentage differences can still appear from sampling and from the latent factor used in simulation.


In [ ]:
for col in ['department', 'job_level', 'education']:
    print('====', col, 'counts ====')
    print(df[col].value_counts())
    print()
    print(pd.crosstab(df[col], df['perf_band_next'], normalize='index').mul(100).round(2))
    print()


## Review-cycle comparisons

Each employee has two consecutive cycles. Cycle volumes are uneven because start cycles were assigned at random.


In [ ]:
print(df['review_cycle'].value_counts().sort_index())
print()
print(pd.crosstab(df['review_cycle'], df['perf_band_next'], normalize='index').mul(100).round(2))
cycle_order = sorted(df['review_cycle'].unique())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=df, x='review_cycle', order=cycle_order, ax=axes[0])
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_title('Rows by review cycle')
pd.crosstab(df['review_cycle'], df['perf_band_next'], normalize='index').loc[cycle_order].plot(
    kind='bar', stacked=True, ax=axes[1]
)
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_title('Band mix by cycle (row %)')
axes[1].set_ylabel('proportion')
fig.tight_layout()
plt.show()


## Correlation analysis (numeric columns only)

employee_id is excluded: it is not numeric and is not a meaningful correlation input.

Pearson **r** measures linear association. It does **not** mean one column causes another.


In [ ]:
numeric = df.select_dtypes(include='number')
corr = numeric.corr(numeric_only=True)
print('Highest |r| pairs:')
pairs = []
cols = list(corr.columns)
for i, c1 in enumerate(cols):
    for c2 in cols[i + 1:]:
        pairs.append((abs(corr.loc[c1, c2]), corr.loc[c1, c2], c1, c2))
for _, r, c1, c2 in sorted(pairs, reverse=True)[:12]:
    print(f'{c1:28s} vs {c2:28s}  r={r:7.4f}')

plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap='vlag', center=0, vmin=-1, vmax=1, square=True)
plt.title('Pearson correlation (numeric columns, synthetic)')
plt.show()


## Written observations (from the Phase 3 executed run)

These figures were calculated on the clean 2,000-row file. Re-run the cells above to refresh them.

1. **Observation:** Medium 743 (37.15%), High 671 (33.55%), Low 586 (29.30%). **Interpretation:** Mild imbalance; still use macro-F1 later. Not a real org rating curve.

2. **Observation:** Mean on-time rate 0.860 High vs 0.674 Low; mean delay 3.245 vs 6.459 days. Boxes overlap. **Interpretation:** Matches the published label formula plus noise, not independent HR proof.

3. **Observation:** Mean sick_days ~7.57 High vs ~7.57 Low; login hours ~8.1 across bands. **Interpretation:** Health/activity fields should not be treated as performance. Prefer excluding them in Phase 4.

4. **Observation:** Engineering has 570 rows; High share ranges from Operations 38.32% to Customer Success 29.52%. Department is not in the label formula. **Interpretation:** Do not rank real teams from this chart.

5. **Observation:** Strongest Pearson pair among numerics is on-time vs code_review_score r=0.639. Delay vs on-time r=-0.547. **Interpretation:** Shared latent factor. Correlation is not causation.

6. **Observation:** 516 / 1,000 employees (51.6%) change band across their two cycles. **Interpretation:** The target is cycle-specific with noise. Do not use employee_id as a feature; prefer grouped validation.

See docs/eda_findings.md for the full write-up.


## Synthetic-data limitations — not real HR evidence

- Labels come from a documented weighted sum + Gaussian noise (src/generate_data.py), not from managers or business outcomes.
- Recovering those weights in EDA is expected. It does **not** validate an employee-evaluation product.
- No gender, race, religion, or other protected attributes are in the file. Their absence does not make a future model fair to use on people.
- Do not auto-promote, fire, or discipline anyone from these scores — including in demo copy.
- Random row splits can leak because each person appears twice.

**Next (Phase 4, not in this notebook):** leakage-safe Logistic Regression vs Random Forest comparison.
